In [6]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 27.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 58.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [11]:
import sys, os
# Add the parent folder (repo root) to sys.path
sys.path.insert(0, os.path.abspath(".."))

from pathlib import Path
import os

# Resolve paths from the repo root (parent of notebooks/)
ROOT = Path.cwd().parent
RAW  = os.getenv("DATA_DIR_RAW") or str(ROOT / "data" / "raw")
PROC = os.getenv("DATA_DIR_PROCESSED") or str(ROOT / "data" / "processed")

# Make sure directories exist before listing/using them
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

print("ROOT:", ROOT)
print("RAW :", RAW)
print("PROC:", PROC)

ROOT: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework
RAW : /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/data/raw
PROC: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/data/processed


In [12]:
import os, shutil
from glob import glob
from datetime import datetime
import pandas as pd

# sanity prints
print("CWD:", os.getcwd())
print("RAW:", RAW)
print("Files currently in RAW:", os.listdir(RAW))

# 1) try to find any CSV in data/raw
files = sorted(glob(os.path.join(RAW, "*.csv")))
raw_file = None

# 2) if none, try copying data/starter_data.csv into data/raw
starter = os.path.join("data", "starter_data.csv")
if not files and os.path.exists(starter):
    ts = datetime.now().strftime("%Y%m%d-%H%M")
    raw_file = os.path.join(RAW, f"starter_{ts}.csv")
    shutil.copy2(starter, raw_file)
    print("Copied starter_data.csv ->", raw_file)

# 3) if still none, pull AAPL via yfinance and save
if not files and raw_file is None:
    print("No CSVs in RAW and no starter found — pulling AAPL via yfinance…")
    import yfinance as yf
    df_tmp = yf.download("AAPL", start="2024-01-01", end="2024-08-01")
    df_tmp.reset_index(inplace=True)
    ts = datetime.now().strftime("%Y%m%d-%H%M")
    raw_file = os.path.join(RAW, f"api_yahoo_AAPL_{ts}.csv")
    df_tmp.to_csv(raw_file, index=False)
    print("Saved:", raw_file)

# finalize raw_file path
if raw_file is None:
    raw_file = files[-1]  # latest from data/raw
print("Using raw file:", raw_file)

# ---- load & clean ----
df_raw = pd.read_csv(raw_file)
df_filled  = fill_missing_median(df_raw)
df_dropped = drop_missing(df_filled, threshold=0.5)
df_cleaned = normalize_data(df_dropped)

# save cleaned
ts = datetime.now().strftime("%Y%m%d-%H%M")
clean_path = os.path.join(PROC, f"cleaned_{ts}.csv")
df_cleaned.to_csv(clean_path, index=False)
print("✅ Cleaned saved:", clean_path)

# quick compare
print("Original shape:", df_raw.shape, " | Cleaned shape:", df_cleaned.shape)
print("Missing before (head):\n", df_raw.isna().sum().head())
print("Missing after (head):\n", df_cleaned.isna().sum().head())

CWD: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/notebooks
RAW: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/data/raw
Files currently in RAW: ['api_yahoo_AAPL_20250820-2333.csv', 'scrape_wikipedia_sp500_20250820-2335.csv']
Using raw file: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/data/raw/scrape_wikipedia_sp500_20250820-2335.csv
✅ Cleaned saved: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/data/processed/cleaned_20250821-0004.csv
Original shape: (503, 8)  | Cleaned shape: (503, 8)
Missing before (head):
 Symbol                   0
Security                 0
GICS Sector              0
GICS Sub-Industry        0
Headquarters Location    0
dtype: int64
Missing after (head):
 Symbol                   0
Security                 0
GICS Sector              0
GICS Sub-Industry        0
Headquarters Location    0
dtype: int64
